# Synthetic data


In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/report.csv")
pd.set_option("display.max_rows", None)
display(df.T)

pd.reset_option("display.max_columns")
pd.reset_option("display.width")

,0
Organization,MBMR1
Point of Care,RAD MR MB
Source System,EPIC
Accession Number,20031798852
Modality,MRI
Exam Code,MBRUE
Exam Description,MR BRAIN WITH AND WITHOUT CONTRAST
CPT Code,70553
Report Text,MR BRAIN WITH AND WITHOUT CONTRAST: 5/15/2023...
Is Stat,False


## Synthetic data generation with OpenAI


In [ ]:
import csv
import io
import json
import os
from typing import Any, Dict, List

import openai
import pandas as pd
from tqdm import tqdm

API_KEY = "YOUR_API_KEY"
API_VERSION = "2024-10-21"
RESOURCE_ENDPOINT = "ENDPOINT_NAME"
MODEL = "gpt-4o-2024-08-06"

client = openai.AzureOpenAI(
    api_key=API_KEY,
    api_version=API_VERSION,
    azure_endpoint=RESOURCE_ENDPOINT,
)


def generate_synthetic_data(
    sample_data: pd.DataFrame, context_info: Dict[str, Any], num_samples: int = 2
) -> List[Dict[str, Any]]:
    """
    Generate synthetic data based on sample data while preserving privacy and data integrity.

    Args:
        sample_data (pd.DataFrame): Sample data to base synthetic data on
        context_info (Dict[str, Any]): Context information for generation (e.g., data_type, specific_context)
        num_samples (int): Number of synthetic samples to generate

    Returns:
        List[Dict[str, Any]]: List of dictionaries containing synthetic data records
    """
    data_type = context_info.get("data_type", "mri report")
    specific_context = context_info.get("specific_context", "")

    min_examples = 2
    real_examples = sample_data.sample(min(min_examples, len(sample_data))).to_dict(
        "records"
    )

    system_prompt = f"""You are a data synthesis expert specializing in medical {data_type} generation.
    Your task is to generate synthetic data that closely resembles the provided examples
    while ensuring complete privacy protection. The synthetic data should maintain the characteristics, 
    terminology, and structure of the real data but contain no actual identifiable information.
    
    Pay careful attention to the content of the example data to determine the appropriate medical domain and terminology.
    """

    user_prompt = f"""Given the following context: {specific_context}
    Generate {num_samples} synthetic {data_type} records based on these real examples:
    
    {json.dumps(real_examples, indent=2)}
    
    Requirements:
    1. Each record must follow the exact format of the examples
    2. Use realistic but synthetic identifiers
    3. Maintain the structure, length, and detail level of real data
    4. Carefully analyze the examples to identify the medical domain and use appropriate terminology
    5. ALL fields must have valid values - no empty fields
    
    Return your response in this exact JSON format:
    {{
    "records": [
        # {num_samples} records here with all fields
    ]
    }}
    """

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.5,
        max_tokens=4000,
        response_format={"type": "json_object"},
    )

    # Add this debug code
    content_str = response.choices[0].message.content
    print(f"Raw API response for {data_type}:\n{content_str[:200]}...")

    try:
        # Parse response
        content = json.loads(content_str)

        # Debug the structure
        print(f"Response type: {type(content)}")
        print(
            f"Keys if dict: {content.keys() if isinstance(content, dict) else 'Not a dict'}"
        )

        # Extract records (handle different possible response formats)
        if isinstance(content, dict) and "records" in content:
            records = content["records"]
        elif isinstance(content, dict) and "data" in content:
            records = content["data"]
        elif isinstance(content, list):
            records = content
        else:
            # Try to find any list in the response
            for key, value in content.items():
                if isinstance(value, list) and len(value) > 0:
                    records = value
                    break
            else:
                records = [content]

        return records

    except Exception as e:
        print(f"Error generating synthetic data: {str(e)}")
        # Return empty list as fallback
        return []


In [ ]:
def get_cancer_type(filename: str) -> str:
    """Extract and standardize cancer type from filename."""
    cancer_type = filename.replace("search_", "").split("_")[0]
    return cancer_type.lower()


# files in the system
CANCER_FILES = [
    "search_astrocytoma_MR_perReport.xlsx",
    "search_Ependymoma_MR_perReport.xlsx",
    "search_Ganglioglioma_MR_perReport.xlsx",
    "search_glioblastoma_MR_perReport.xlsx",
    "search_Gliomatosis_MR_perReport.xlsx",
    "search_Gliosarcoma_MR_perReport.xlsx",
    "search_oligoastrocytoma_MR_perReport.xlsx",
    "search_oligodendroglioma_MR_perReport.xlsx",
    "search_Xanthoastrocytoma_MR_perReport.xlsx",
]

COLUMNS_NEEDED = [
    "Patient MRN",
    "Report Text",
    "Exam Description",
    "Patient Sex",
    "Patient Age",
    "Patient Status",
]

# Base directory with MRI and pathology folders
base_dir = "cancer_data"
mri_dir = os.path.join(base_dir, "mri")
path_dir = os.path.join(base_dir, "pathology")

os.makedirs(mri_dir, exist_ok=True)
os.makedirs(path_dir, exist_ok=True)


def process_synthetic_data(
    raw_records: List[Dict[str, Any]], sample_data: pd.DataFrame
) -> pd.DataFrame:
    """
    Process raw synthetic records into a DataFrame with the same structure as the sample data.

    Args:
        raw_records (List[Dict[str, Any]]): Raw synthetic records
        sample_data (pd.DataFrame): Original sample data with desired structure

    Returns:
        pd.DataFrame: Processed synthetic data
    """
    # Convert to DataFrame
    df = pd.DataFrame(raw_records)

    # Ensure all required columns exist
    for col in sample_data.columns:
        if col not in df.columns:
            df[col] = ""

    # Reorder columns to match original
    df = df[sample_data.columns]

    # Clean text fields
    if "Report Text" in df.columns:
        df["Report Text"] = df["Report Text"].apply(lambda x: str(x).strip())
    if "Exam Description" in df.columns:
        df["Exam Description"] = df["Exam Description"].apply(lambda x: str(x).strip())

    # Ensure numeric fields are numeric
    if "Patient Age" in df.columns:
        df["Patient Age"] = (
            pd.to_numeric(df["Patient Age"], errors="coerce").fillna(50).astype(int)
        )
    if "Patient MRN" in df.columns:
        df["Patient MRN"] = (
            pd.to_numeric(df["Patient MRN"], errors="coerce").fillna(10000).astype(int)
        )

    return df


base_path = "/data/users/neob/mPower/perReport_12122024"

for filename in tqdm(CANCER_FILES, desc="Processing cancer files"):
    cancer_type = get_cancer_type(filename)
    print(f"\nProcessing {cancer_type}...")

    # Define file paths
    mri_file = f"{base_path}/{filename}"
    path_filename = filename.replace("_MR_", "_pathology_")
    path_file = f"{base_path}/{path_filename}"

    mri_output_path = f"{mri_dir}/{cancer_type}_synthetic.csv"
    path_output_path = f"{path_dir}/{cancer_type}_synthetic.csv"

    # Process MRI data
    try:
        # Load only needed columns to optimize memory
        mri_df = pd.read_excel(mri_file, usecols=COLUMNS_NEEDED, nrows=20)

        # Generate synthetic MRI data with enhanced context
        context_info = {
            "data_type": "mri",
            "specific_context": f"Brain MRI reports for {cancer_type} patients. These reports typically describe tumor location, size, enhancement patterns, and surrounding edema. {cancer_type} typically presents with specific imaging characteristics that should be reflected in the synthetic data.",
        }
        synthetic_mri_records = generate_synthetic_data(mri_df, context_info)

        # Process into DataFrame with correct structure
        synthetic_mri_df = process_synthetic_data(synthetic_mri_records, mri_df)

        # Write to CSV
        synthetic_mri_df.to_csv(mri_output_path, index=False)
        print(
            f"Generated {len(synthetic_mri_df)} synthetic MRI records for {cancer_type}"
        )
    except Exception as e:
        print(f"Error processing MRI file for {cancer_type}: {str(e)}")

    # Process pathology data
    try:
        path_df = pd.read_excel(path_file, usecols=COLUMNS_NEEDED, nrows=20)

        # Generate synthetic pathology data with enhanced context
        context_info = {
            "data_type": "pathology",
            "specific_context": f"Pathology reports for {cancer_type} brain tumors. These reports typically include histological features, immunohistochemistry results, molecular markers, and WHO grading specific to {cancer_type}. The synthetic data should maintain the specialized terminology and structure of neuropathology reports.",
        }
        synthetic_path_records = generate_synthetic_data(path_df, context_info)

        # Process into DataFrame with correct structure
        synthetic_path_df = process_synthetic_data(synthetic_path_records, path_df)

        # Write to CSV
        synthetic_path_df.to_csv(path_output_path, index=False)
        print(
            f"Generated {len(synthetic_path_df)} synthetic pathology records for {cancer_type}"
        )
    except Exception as e:
        print(f"Error processing pathology file for {cancer_type}: {str(e)}")


In [ ]:
train_df = pd.read_csv("/home/neob/ai/paper/data/training/mnrl/train.csv")
synthetic_train = generate_synthetic_data(
    sample_data=train_df.head(100),
    context_info={
        "data_type": "MRI and pathology",
        "specific_context": f"MNRL training pairs for brain cancer reports, cancer types must be one of the following: {train_df.cancer_type.unique()}",
    },
    num_samples=1,
)
pd.DataFrame(synthetic_train).to_csv("synthetic_pairs.csv", index=False)

In [ ]:
matches_df = pd.read_csv("/home/neob/ai/paper/data/matches.csv")
synthetic_matches = generate_synthetic_data(
    sample_data=matches_df[["mrn", "cancer_type", "mri_text", "path_text"]].head(10),
    context_info={
        "data_type": "mri and pathology",
        "specific_context": f"Brain cancer MRI and pathology report matches, cancer_type must be one of the following {matches_df.cancer_type.unique()}",
    },
    num_samples=2,
)
pd.DataFrame(synthetic_matches).to_csv("synthetic_matches.csv", index=False)

In [ ]:
tumor_type_eval_df = pd.read_csv("/home/neob/ai/paper/data/eval/tumor_type_eval.csv")

synthetic_tumor_type = generate_synthetic_data(
    sample_data=tumor_type_eval_df[["MRN", "report", "tumor_type"]].head(100),
    context_info={
        "data_type": "MRI",
        "specific_context": f"Tumor type evaluation dataset, cancer type must be one of the following {tumor_type_eval_df.tumor_type.unique()}",
    },
    num_samples=5,
)

pd.DataFrame(synthetic_tumor_type).to_csv("synthetic_tumor_type_eval.csv", index=False)

In [ ]:
mgmt_eval_df = pd.read_csv("/home/neob/ai/paper/data/eval/mgmt_eval.csv")
synthetic_mgmt = generate_synthetic_data(
    sample_data=mgmt_eval_df[["mrn", "report", "mgmt"]].head(5),
    context_info={
        "data_type": "pathology",
        "specific_context": "MGMT status evaluation dataset",
    },
    num_samples=5,
)

pd.DataFrame(synthetic_mgmt).to_csv("synthetic_mgmt_eval.csv", index=False)

In [ ]:
os_eval = pd.read_csv("/home/neob/ai/paper/data/eval/os_eval.csv")
synthetic_mgmt = generate_synthetic_data(
    sample_data=os_eval.head(100),
    context_info={
        "data_type": "MRI",
        "specific_context": "overall survival dataset of glioblastoma patients",
    },
    num_samples=2,
)

pd.DataFrame(synthetic_mgmt).to_csv("synthetic_os_eval.csv", index=False)